# 3-Way Yield Curve Comparison
## Historical vs Diffusion-Generated vs Traditional HJM+PCA

---

This notebook performs a comprehensive statistical comparison of three yield curve datasets:

1. **Historical** — Real market data
2. **Diffusion** — Generated using diffusion models
3. **Traditional HJM+PCA** — Generated using Heath-Jarrow-Morton with PCA

### Metrics Analyzed:
- PCA structure (explained variance, loadings)
- Volatility term structure
- Cross-maturity correlation
- Lag-1 autocorrelation
- Wasserstein distances between distributions (in shared PCA space)

### Output:
- CSV files with detailed metrics
- PDF report with visualizations

---
## 1. Configuration

**⚠️ IMPORTANT: Replace these template filenames with your actual data files**

In [ ]:
import os
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.stats import wasserstein_distance
from matplotlib.backends.backend_pdf import PdfPages

# ============================================================
# FILE PATHS - REPLACE THESE WITH YOUR ACTUAL FILES
# ============================================================

HISTORICAL_FILE = "raw_macro_yield_train(in).csv"
DIFFUSION_FILE = "generated_autogressive_yield_scenarios.csv" #need for adjustments 12 -> 60 timesteps
TRADITIONAL_FILE = "hjm_pca_all_generations.csv"

# Output configuration
OUTPUT_DIR = "3way_yield_curve_analysis"
OUTPUT_PDF = os.path.join(OUTPUT_DIR, "3way_comparison_report.pdf")

# Yield columns (standard US Treasury maturities)
YIELD_COLS = [
    "Yield_1M", "Yield_3M", "Yield_6M", "Yield_1Y", "Yield_2Y",
    "Yield_3Y", "Yield_5Y", "Yield_7Y", "Yield_10Y", "Yield_20Y", "Yield_30Y"
]

MATURITY_LABELS = ["1M", "3M", "6M", "1Y", "2Y", "3Y", "5Y", "7Y", "10Y", "20Y", "30Y"]

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Configuration loaded.")
print(f"Output directory: {OUTPUT_DIR}")

Configuration loaded.
Output directory: 3way_yield_curve_analysis

⚠️  Make sure to replace template filenames with actual data files!


---
## 2. Helper Functions

In [2]:
def add_text_page(pdf, title, lines, fontsize=10, lines_per_page=35):
    """Add text page(s) to PDF report."""
    chunks = [lines[i:i + lines_per_page] for i in range(0, len(lines), lines_per_page)]
    if not chunks:
        chunks = [[]]

    for page_num, chunk in enumerate(chunks, start=1):
        fig = plt.figure(figsize=(11, 8.5))
        plt.axis("off")

        full_title = title if len(chunks) == 1 else f"{title} (page {page_num})"
        plt.text(0.01, 0.97, full_title, fontsize=14, fontweight="bold", va="top", ha="left")

        y = 0.92
        for line in chunk:
            plt.text(0.01, y, line, fontsize=fontsize, family="monospace", va="top", ha="left")
            y -= 0.025

        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)


def series_to_lines(title, s, float_fmt="{:,.6f}"):
    """Convert pandas Series to formatted text lines."""
    lines = [title, "-" * len(title)]
    for idx, val in s.items():
        if pd.isna(val):
            lines.append(f"{idx:<30} NaN")
        else:
            lines.append(f"{idx:<30} {float_fmt.format(val)}")
    return lines


def df_to_lines(title, df, float_fmt="{:,.6f}"):
    """Convert pandas DataFrame to formatted text lines."""
    lines = [title, "-" * len(title)]
    lines.append(df.to_string(float_format=lambda x: float_fmt.format(x)))
    return lines


def plot_and_save(pdf, fig):
    """Save figure to PDF and close."""
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)


print("Helper functions defined.")

Helper functions defined.


---
## 3. Data Loading & Preprocessing

In [3]:
def preprocess_dataset(data: pd.DataFrame, yield_cols: list, dataset_name: str) -> pd.DataFrame:
    """
    Extract yield columns and remove rows with missing values.
    
    Parameters:
    -----------
    data : DataFrame with yield columns
    yield_cols : list of column names to extract
    dataset_name : name for error messages
    
    Returns:
    --------
    DataFrame with only yield columns, no missing values
    """
    missing = [c for c in yield_cols if c not in data.columns]
    if missing:
        raise ValueError(f"{dataset_name} is missing columns: {missing}")
    
    yields = data[yield_cols].dropna().copy()
    
    if len(yields) == 0:
        raise ValueError(f"{dataset_name} has no valid rows after removing NaN values")
    
    return yields


# Load all three datasets
print("Loading datasets...")

historical_data = pd.read_csv(HISTORICAL_FILE)
print(f"✓ Historical: {len(historical_data):,} rows")

diffusion_data = pd.read_csv(DIFFUSION_FILE)
print(f"✓ Diffusion: {len(diffusion_data):,} rows")

traditional_data = pd.read_csv(TRADITIONAL_FILE)
print(f"✓ Traditional: {len(traditional_data):,} rows")

print("\nAll datasets loaded successfully.")

Loading datasets...
✓ Historical: 4,836 rows
✓ Diffusion: 6,500 rows
✓ Traditional: 30,500 rows

All datasets loaded successfully.


---
## 4. Statistical Analysis Functions

In [4]:
def analyze_dataset(data: pd.DataFrame, dataset_name: str, yield_cols: list):
    """
    Perform complete statistical analysis on a single dataset.
    
    Returns dictionary with:
    - PCA results (loadings, scores, explained variance)
    - Volatility term structure
    - Cross-maturity correlation
    - Lag-1 autocorrelation
    """
    print(f"\nAnalyzing {dataset_name}...")
    
    # Preprocess
    yields = preprocess_dataset(data, yield_cols, dataset_name)
    print(f"  Valid observations: {len(yields):,}")
    
    # Standardize for PCA
    z = (yields - yields.mean()) / yields.std(ddof=1)
    
    # PCA with 3 components
    pca = PCA(n_components=3)
    scores_array = pca.fit_transform(z)
    scores = pd.DataFrame(scores_array, columns=["PC1", "PC2", "PC3"], index=yields.index)
    
    # Explained variance
    explained_variance = pd.Series(
        pca.explained_variance_ratio_,
        index=["PC1", "PC2", "PC3"],
        name=f"{dataset_name}_explained_variance"
    )
    
    # PCA loadings
    loadings = pd.DataFrame(
        pca.components_.T,
        index=yield_cols,
        columns=["PC1", "PC2", "PC3"]
    )
    
    # Score correlations
    score_corr = scores.corr()
    
    # Volatility (standard deviation of each maturity)
    volatility = yields.std(ddof=1)
    
    # Cross-maturity correlation
    cross_corr = yields.corr()
    
    # Lag-1 autocorrelation
    autocorr = pd.Series(
        {col: yields[col].autocorr(lag=1) for col in yield_cols},
        name=f"{dataset_name}_lag1_autocorr"
    )
    
    print(f"  ✓ PCA complete: {explained_variance.sum()*100:.2f}% variance in 3 PCs")
    print(f"  ✓ Volatility range: {volatility.min():.4f} - {volatility.max():.4f}")
    
    return {
        "dataset_name": dataset_name,
        "raw_data": data,
        "yields": yields,
        "z": z,
        "pca_model": pca,
        "scores": scores,
        "explained_variance": explained_variance,
        "cumulative_variance_3pc": explained_variance.sum(),
        "loadings": loadings,
        "score_corr": score_corr,
        "volatility": volatility,
        "cross_corr": cross_corr,
        "autocorr": autocorr,
    }


print("Analysis function defined.")

Analysis function defined.


---
## 5. Pairwise Comparison Functions

In [5]:
def compare_to_historical(hist_results, other_results):
    """
    Compare another dataset to historical baseline.
    Computes differences in key metrics.
    """
    other_name = other_results["dataset_name"]
    
    # Explained variance comparison
    explained_comp = pd.DataFrame({
        "Historical": hist_results["explained_variance"],
        other_name: other_results["explained_variance"]
    })
    explained_comp["Difference"] = explained_comp[other_name] - explained_comp["Historical"]
    
    # Volatility comparison
    volatility_comp = pd.DataFrame({
        "Historical": hist_results["volatility"],
        other_name: other_results["volatility"]
    })
    volatility_comp["Difference"] = volatility_comp[other_name] - volatility_comp["Historical"]
    volatility_comp["Abs_Difference"] = volatility_comp["Difference"].abs()
    
    # Autocorrelation comparison
    autocorr_comp = pd.DataFrame({
        "Historical": hist_results["autocorr"],
        other_name: other_results["autocorr"]
    })
    autocorr_comp["Difference"] = autocorr_comp[other_name] - autocorr_comp["Historical"]
    autocorr_comp["Abs_Difference"] = autocorr_comp["Difference"].abs()
    
    # Cross-correlation difference
    corr_diff = other_results["cross_corr"] - hist_results["cross_corr"]
    
    return {
        "explained_comparison": explained_comp,
        "volatility_comparison": volatility_comp,
        "autocorr_comparison": autocorr_comp,
        "corr_diff": corr_diff,
    }


print("Comparison function defined.")

Comparison function defined.


---
## 6. Shared PCA Basis (Historical Reference)

In [6]:
def build_shared_historical_basis(all_results: dict, yield_cols: list, reference_name="Historical"):
    """
    Create a common PCA basis using historical data as reference.
    Project all datasets into this shared space for fair comparison.
    
    This ensures that differences in PCA scores reflect actual distributional
    differences, not just different PCA orientations.
    """
    print(f"\nBuilding shared PCA basis using {reference_name} as reference...")
    
    # Fit scaler and PCA on historical data only
    scaler = StandardScaler()
    pca = PCA(n_components=3)
    
    hist_y = all_results[reference_name]["yields"][yield_cols]
    hist_z = scaler.fit_transform(hist_y)
    pca.fit(hist_z)
    
    # Project all datasets into this common space
    shared_scores = {}
    for name, r in all_results.items():
        z = scaler.transform(r["yields"][yield_cols])
        shared_scores[name] = pd.DataFrame(
            pca.transform(z),
            columns=["PC1", "PC2", "PC3"]
        )
        print(f"  ✓ Projected {name}: {len(shared_scores[name])} observations")
    
    # Get shared loadings
    shared_loadings = pd.DataFrame(
        pca.components_.T,
        index=yield_cols,
        columns=["PC1", "PC2", "PC3"]
    )
    
    return {
        "scaler": scaler,
        "pca": pca,
        "shared_loadings": shared_loadings,
        "scores": shared_scores,
    }


print("Shared basis function defined.")

Shared basis function defined.


---
## 7. Wasserstein Distance Calculation

In [7]:
def compute_pairwise_wasserstein(shared_scores: dict):
    """
    Compute Wasserstein distances between all pairs of datasets.
    
    Wasserstein distance (Earth Mover's Distance) measures how different
    two probability distributions are. Lower values = more similar.
    """
    print("\nComputing pairwise Wasserstein distances...")
    
    rows = []
    keys = list(shared_scores.keys())
    
    for a, b in itertools.combinations(keys, 2):
        d1 = wasserstein_distance(shared_scores[a]["PC1"], shared_scores[b]["PC1"])
        d2 = wasserstein_distance(shared_scores[a]["PC2"], shared_scores[b]["PC2"])
        d3 = wasserstein_distance(shared_scores[a]["PC3"], shared_scores[b]["PC3"])
        avg = np.mean([d1, d2, d3])
        
        rows.append({
            "Pair": f"{a} vs {b}",
            "PC1": d1,
            "PC2": d2,
            "PC3": d3,
            "Average": avg,
        })
        
        print(f"  {a} vs {b}: avg = {avg:.6f}")
    
    return pd.DataFrame(rows).set_index("Pair")


print("Wasserstein function defined.")

Wasserstein function defined.


---
## 8. Run Complete Analysis

In [8]:
# Analyze all three datasets
all_results = {
    "Historical": analyze_dataset(historical_data, "Historical", YIELD_COLS),
    "Diffusion": analyze_dataset(diffusion_data, "Diffusion", YIELD_COLS),
    "Traditional_HJM_PCA": analyze_dataset(traditional_data, "Traditional_HJM_PCA", YIELD_COLS),
}

# Pairwise comparisons vs historical
comparisons_vs_hist = {
    "Diffusion": compare_to_historical(all_results["Historical"], all_results["Diffusion"]),
    "Traditional_HJM_PCA": compare_to_historical(all_results["Historical"], all_results["Traditional_HJM_PCA"]),
}

# Shared PCA basis
shared_basis = build_shared_historical_basis(all_results, YIELD_COLS, reference_name="Historical")

# Wasserstein distances
pairwise_wass = compute_pairwise_wasserstein(shared_basis["scores"])

print("\n" + "="*60)
print("ANALYSIS COMPLETE")
print("="*60)


Analyzing Historical...
  Valid observations: 4,836
  ✓ PCA complete: 99.78% variance in 3 PCs
  ✓ Volatility range: 1.1046 - 1.5227

Analyzing Diffusion...
  Valid observations: 6,500
  ✓ PCA complete: 96.89% variance in 3 PCs
  ✓ Volatility range: 0.5454 - 2.5260

Analyzing Traditional_HJM_PCA...
  Valid observations: 30,500
  ✓ PCA complete: 99.90% variance in 3 PCs
  ✓ Volatility range: 1.3886 - 1.6196

Building shared PCA basis using Historical as reference...
  ✓ Projected Historical: 4836 observations
  ✓ Projected Diffusion: 6500 observations
  ✓ Projected Traditional_HJM_PCA: 30500 observations

Computing pairwise Wasserstein distances...
  Historical vs Diffusion: avg = 5.353052
  Historical vs Traditional_HJM_PCA: avg = 1.824356
  Diffusion vs Traditional_HJM_PCA: avg = 3.542920

ANALYSIS COMPLETE


/Users/kaat/Desktop/Thesis Repository/Thesis-ALM/.venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: divide by zero encountered in matmul
  X_transformed = X @ self.components_.T
/Users/kaat/Desktop/Thesis Repository/Thesis-ALM/.venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: overflow encountered in matmul
  X_transformed = X @ self.components_.T
/Users/kaat/Desktop/Thesis Repository/Thesis-ALM/.venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: invalid value encountered in matmul
  X_transformed = X @ self.components_.T
/Users/kaat/Desktop/Thesis Repository/Thesis-ALM/.venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: divide by zero encountered in matmul
  X_transformed = X @ self.components_.T
/Users/kaat/Desktop/Thesis Repository/Thesis-ALM/.venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: overflow encountered

---
## 9. Display Key Results

In [9]:
print("\n" + "="*60)
print("3-WAY COMPARISON SUMMARY")
print("="*60)

# Dataset sizes
print("\nDataset Sizes:")
for name, r in all_results.items():
    print(f"  {name:25s}: {len(r['yields']):>8,} observations")

# Explained variance
print("\nExplained Variance (3 PCs):")
explained_3way = pd.DataFrame({
    name: r["explained_variance"] for name, r in all_results.items()
})
print(explained_3way)
print("\nCumulative:")
for name, r in all_results.items():
    print(f"  {name:25s}: {r['cumulative_variance_3pc']*100:>6.2f}%")

# Wasserstein distances
print("\nWasserstein Distances (Shared Historical PCA Basis):")
print(pairwise_wass)

print("\n" + "="*60)


3-WAY COMPARISON SUMMARY

Dataset Sizes:
  Historical               :    4,836 observations
  Diffusion                :    6,500 observations
  Traditional_HJM_PCA      :   30,500 observations

Explained Variance (3 PCs):
     Historical  Diffusion  Traditional_HJM_PCA
PC1    0.841005   0.864520             0.918440
PC2    0.148156   0.068337             0.074908
PC3    0.008618   0.036063             0.005647

Cumulative:
  Historical               :  99.78%
  Diffusion                :  96.89%
  Traditional_HJM_PCA      :  99.90%

Wasserstein Distances (Shared Historical PCA Basis):
                                         PC1       PC2       PC3   Average
Pair                                                                      
Historical vs Diffusion            10.442167  4.794399  0.822590  5.353052
Historical vs Traditional_HJM_PCA   4.810275  0.482695  0.180099  1.824356
Diffusion vs Traditional_HJM_PCA    5.655185  4.330939  0.642636  3.542920



---
## 10. Save Results to CSV

In [10]:
def save_results_to_csv(all_results, comparisons_vs_hist, shared_basis, pairwise_wass, output_dir):
    """Save all analysis results as CSV files."""
    print("\nSaving results to CSV...")
    
    # Per-dataset outputs
    for name, r in all_results.items():
        key = name.lower().replace(" ", "_").replace("+", "plus")
        
        r["explained_variance"].to_csv(os.path.join(output_dir, f"{key}_explained_variance.csv"))
        r["loadings"].to_csv(os.path.join(output_dir, f"{key}_pca_loadings.csv"))
        r["scores"].describe().to_csv(os.path.join(output_dir, f"{key}_score_statistics.csv"))
        r["score_corr"].to_csv(os.path.join(output_dir, f"{key}_score_correlation.csv"))
        r["volatility"].to_csv(os.path.join(output_dir, f"{key}_volatility.csv"))
        r["cross_corr"].to_csv(os.path.join(output_dir, f"{key}_cross_maturity_correlation.csv"))
        r["autocorr"].to_csv(os.path.join(output_dir, f"{key}_lag1_autocorrelation.csv"))
        
        print(f"  ✓ {name} metrics saved")
    
    # Historical-vs-others comparison outputs
    for name, c in comparisons_vs_hist.items():
        key = name.lower().replace(" ", "_").replace("+", "plus")
        
        c["explained_comparison"].to_csv(os.path.join(output_dir, f"hist_vs_{key}_explained_variance.csv"))
        c["volatility_comparison"].to_csv(os.path.join(output_dir, f"hist_vs_{key}_volatility.csv"))
        c["autocorr_comparison"].to_csv(os.path.join(output_dir, f"hist_vs_{key}_autocorrelation.csv"))
        c["corr_diff"].to_csv(os.path.join(output_dir, f"hist_vs_{key}_cross_corr_diff.csv"))
        
        print(f"  ✓ Historical vs {name} comparison saved")
    
    # Shared basis outputs
    shared_basis["shared_loadings"].to_csv(os.path.join(output_dir, "shared_pca_loadings.csv"))
    
    for name, df_scores in shared_basis["scores"].items():
        key = name.lower().replace(" ", "_").replace("+", "plus")
        df_scores.describe().to_csv(os.path.join(output_dir, f"{key}_shared_basis_scores.csv"))
    
    # Wasserstein distances
    pairwise_wass.to_csv(os.path.join(output_dir, "pairwise_wasserstein_distances.csv"))
    
    print(f"  ✓ Shared basis metrics saved")
    print(f"\nAll CSV files saved to: {output_dir}/")


save_results_to_csv(
    all_results=all_results,
    comparisons_vs_hist=comparisons_vs_hist,
    shared_basis=shared_basis,
    pairwise_wass=pairwise_wass,
    output_dir=OUTPUT_DIR,
)


Saving results to CSV...
  ✓ Historical metrics saved
  ✓ Diffusion metrics saved
  ✓ Traditional_HJM_PCA metrics saved
  ✓ Historical vs Diffusion comparison saved
  ✓ Historical vs Traditional_HJM_PCA comparison saved
  ✓ Shared basis metrics saved

All CSV files saved to: 3way_yield_curve_analysis/


---
## 11. Generate PDF Report with Visualizations

In [11]:
def generate_pdf_report(
    all_results,
    comparisons_vs_hist,
    shared_basis,
    pairwise_wass,
    maturity_labels,
    output_pdf,
):
    """Generate comprehensive PDF report with all visualizations."""
    
    colors = {
        "Historical": "#1f77b4",
        "Diffusion": "#2ca02c",
        "Traditional_HJM_PCA": "#ff7f0e",
    }
    
    print("\nGenerating PDF report...")
    
    with PdfPages(output_pdf) as pdf:
        # ========================================
        # COVER PAGE
        # ========================================
        summary_lines = [
            "3-WAY YIELD CURVE COMPARISON REPORT",
            "",
            "Datasets:",
            f"  1. Historical:            {all_results['Historical']['yields'].shape[0]:>8,} observations",
            f"  2. Diffusion:             {all_results['Diffusion']['yields'].shape[0]:>8,} observations",
            f"  3. Traditional HJM+PCA:   {all_results['Traditional_HJM_PCA']['yields'].shape[0]:>8,} observations",
            "",
            f"Maturities analyzed: {len(maturity_labels)}",
            f"  {', '.join(maturity_labels)}",
            "",
            "Analysis includes:",
            "  • PCA structure (3 components)",
            "  • Volatility term structure",
            "  • Cross-maturity correlation",
            "  • Lag-1 autocorrelation",
            "  • Wasserstein distances (in shared historical PCA basis)",
            "",
            "Key methodology:",
            "  All cross-dataset comparisons use a SHARED PCA BASIS",
            "  fitted on Historical data only. This ensures that",
            "  distributional differences are not confounded with",
            "  arbitrary PCA rotation differences.",
        ]
        add_text_page(pdf, "Report Summary", summary_lines, fontsize=11)
        
        # ========================================
        # WASSERSTEIN DISTANCES
        # ========================================
        add_text_page(
            pdf,
            "Pairwise Wasserstein Distances (Shared Historical PCA Basis)",
            df_to_lines("Wasserstein Distances", pairwise_wass)
        )
        
        # ========================================
        # EXPLAINED VARIANCE
        # ========================================
        explained_3way = pd.DataFrame({
            name: r["explained_variance"] for name, r in all_results.items()
        })
        add_text_page(
            pdf,
            "Explained Variance by Principal Component",
            df_to_lines("Explained Variance", explained_3way)
        )
        
        # ========================================
        # VOLATILITY & AUTOCORRELATION COMPARISONS
        # ========================================
        for name, comp in comparisons_vs_hist.items():
            add_text_page(
                pdf,
                f"Historical vs {name} — Volatility",
                df_to_lines(f"Volatility Comparison", comp["volatility_comparison"])
            )
            add_text_page(
                pdf,
                f"Historical vs {name} — Autocorrelation",
                df_to_lines(f"Autocorrelation Comparison", comp["autocorr_comparison"])
            )
        
        # ========================================
        # PLOT 1: PCA LOADINGS (per-dataset basis)
        # ========================================
        print("  Generating plot 1/10: PCA loadings...")
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        for i, pc in enumerate(["PC1", "PC2", "PC3"]):
            ax = axes[i]
            for name, r in all_results.items():
                ax.plot(
                    maturity_labels,
                    r["loadings"][pc],
                    marker="o",
                    label=name,
                    color=colors.get(name, None),
                    linewidth=2,
                )
            ax.set_title(f"{pc} Loadings")
            ax.set_xlabel("Maturity")
            ax.set_ylabel("Loading")
            ax.legend(fontsize=8)
            ax.grid(True, alpha=0.3)
        fig.suptitle("PCA Loadings Across Maturities (per-dataset PCA)", fontsize=13)
        plt.tight_layout()
        plot_and_save(pdf, fig)
        
        # ========================================
        # PLOT 2-4: SHARED BASIS SCORE DISTRIBUTIONS
        # ========================================
        for i, pc in enumerate(["PC1", "PC2", "PC3"], start=2):
            print(f"  Generating plot {i}/10: {pc} distribution...")
            fig = plt.figure(figsize=(10, 6))
            for name, scores_df in shared_basis["scores"].items():
                plt.hist(
                    scores_df[pc],
                    bins=50,
                    density=True,
                    alpha=0.5,
                    label=name,
                    color=colors.get(name, None),
                )
            plt.title(f"{pc} Score Distribution (Shared Historical PCA Basis)")
            plt.xlabel("Score")
            plt.ylabel("Density")
            plt.legend()
            plt.grid(True, alpha=0.3)
            plot_and_save(pdf, fig)
        
        # ========================================
        # PLOT 5: VOLATILITY TERM STRUCTURE
        # ========================================
        print(f"  Generating plot 5/10: Volatility...")
        fig = plt.figure(figsize=(10, 6))
        for name, r in all_results.items():
            plt.plot(
                maturity_labels,
                r["volatility"].values,
                marker="o",
                label=name,
                color=colors.get(name, None),
                linewidth=2,
            )
        plt.title("Volatility Term Structure (Standard Deviation)")
        plt.xlabel("Maturity")
        plt.ylabel("Std Dev")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plot_and_save(pdf, fig)
        
        # ========================================
        # PLOT 6-7: VOLATILITY DIFFERENCES
        # ========================================
        for i, (name, comp) in enumerate(comparisons_vs_hist.items(), start=6):
            print(f"  Generating plot {i}/10: Volatility diff ({name})...")
            fig = plt.figure(figsize=(10, 6))
            plt.plot(
                maturity_labels,
                comp["volatility_comparison"]["Abs_Difference"].values,
                marker="o",
                linewidth=2,
                color=colors.get(name, "gray"),
            )
            plt.title(f"Absolute Volatility Difference: |{name} - Historical|")
            plt.xlabel("Maturity")
            plt.ylabel("Absolute Difference")
            plt.grid(True, alpha=0.3)
            plot_and_save(pdf, fig)
        
        # ========================================
        # PLOT 8: LAG-1 AUTOCORRELATION
        # ========================================
        print(f"  Generating plot 8/10: Autocorrelation...")
        fig = plt.figure(figsize=(10, 6))
        for name, r in all_results.items():
            plt.plot(
                maturity_labels,
                r["autocorr"].values,
                marker="o",
                label=name,
                color=colors.get(name, None),
                linewidth=2,
            )
        plt.title("Lag-1 Autocorrelation")
        plt.xlabel("Maturity")
        plt.ylabel("Autocorrelation")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plot_and_save(pdf, fig)
        
        # ========================================
        # PLOT 9-10: CROSS-MATURITY CORRELATION HEATMAPS
        # ========================================
        for i, name in enumerate(["Historical", "Diffusion"], start=9):
            print(f"  Generating plot {i}/10: Correlation heatmap ({name})...")
            fig = plt.figure(figsize=(8, 7))
            plt.imshow(all_results[name]["cross_corr"], aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1)
            plt.colorbar(label="Correlation")
            plt.title(f"{name} Cross-Maturity Correlation")
            plt.xticks(range(len(maturity_labels)), maturity_labels, rotation=45)
            plt.yticks(range(len(maturity_labels)), maturity_labels)
            plt.tight_layout()
            plot_and_save(pdf, fig)
    
    print(f"\n✓ PDF report saved: {output_pdf}")


generate_pdf_report(
    all_results=all_results,
    comparisons_vs_hist=comparisons_vs_hist,
    shared_basis=shared_basis,
    pairwise_wass=pairwise_wass,
    maturity_labels=MATURITY_LABELS,
    output_pdf=OUTPUT_PDF,
)


Generating PDF report...
  Generating plot 1/10: PCA loadings...
  Generating plot 2/10: PC1 distribution...
  Generating plot 3/10: PC2 distribution...
  Generating plot 4/10: PC3 distribution...
  Generating plot 5/10: Volatility...
  Generating plot 6/10: Volatility diff (Diffusion)...
  Generating plot 7/10: Volatility diff (Traditional_HJM_PCA)...
  Generating plot 8/10: Autocorrelation...
  Generating plot 9/10: Correlation heatmap (Historical)...
  Generating plot 10/10: Correlation heatmap (Diffusion)...

✓ PDF report saved: 3way_yield_curve_analysis/3way_comparison_report.pdf


---
## 12. Final Summary

In [12]:
print("\n" + "="*70)
print("3-WAY COMPARISON COMPLETE")
print("="*70)

print("\n📁 Output Files:")
print(f"   Directory: {OUTPUT_DIR}/")
print(f"   PDF Report: {OUTPUT_PDF}")
print(f"   CSV Files: {len(os.listdir(OUTPUT_DIR)) - 1} files")

print("\n📊 Key Findings:")
print("\n1. Wasserstein Distances (lower = more similar):")
print(pairwise_wass[["Average"]].to_string())

print("\n2. Explained Variance Coverage (3 PCs):")
for name, r in all_results.items():
    print(f"   {name:25s}: {r['cumulative_variance_3pc']*100:>6.2f}%")

print("\n3. Volatility Ranges:")
for name, r in all_results.items():
    vol = r['volatility']
    print(f"   {name:25s}: {vol.min():.4f} - {vol.max():.4f}")

print("\n" + "="*70)
print("✓ Analysis complete. Review the PDF report for detailed visualizations.")
print("="*70)


3-WAY COMPARISON COMPLETE

📁 Output Files:
   Directory: 3way_yield_curve_analysis/
   PDF Report: 3way_yield_curve_analysis/3way_comparison_report.pdf
   CSV Files: 34 files

📊 Key Findings:

1. Wasserstein Distances (lower = more similar):
                                    Average
Pair                                       
Historical vs Diffusion            5.353052
Historical vs Traditional_HJM_PCA  1.824356
Diffusion vs Traditional_HJM_PCA   3.542920

2. Explained Variance Coverage (3 PCs):
   Historical               :  99.78%
   Diffusion                :  96.89%
   Traditional_HJM_PCA      :  99.90%

3. Volatility Ranges:
   Historical               : 1.1046 - 1.5227
   Diffusion                : 0.5454 - 2.5260
   Traditional_HJM_PCA      : 1.3886 - 1.6196

✓ Analysis complete. Review the PDF report for detailed visualizations.
